# Fraud model as an API (FastAPI)

The model from notebook 01 is saved in `models/fraud_model.pkl`. Here I serve it with FastAPI and test it.

The API code is in `api/fraud_api.py`. For each payment it:
1. builds the customer key
2. calculates the window features from earlier payments
3. scores the payment with the model
4. returns APPROVE, STEP_UP (OTP) or BLOCK with the top reasons
5. adds the payment to the history

## 1. Load the saved model

In [1]:
import sys
import time
import subprocess

import httpx
import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, "..")

bundle = joblib.load("../models/fraud_model.pkl")
print(bundle["model_name"], "|", len(bundle["features"]), "features")
print("block if score >=", round(bundle["thr_block"], 4))
print("OTP if score >=", round(bundle["thr_review"], 4))

xgboost | 64 features
block if score >= 0.9923
OTP if score >= 0.8427


## 2. The test payments

First 2,000 payments of the test month, with the score the notebook gave them. The API should give the same score.

In [2]:
sample = pd.read_parquet(r"E:\Fraud_Risk_Data\test_sample_2000.parquet")
print(sample.shape)
sample[["TransactionID", "TransactionAmt", "ProductCD", "card4", "card6", "isFraud", "notebook_score"]].head()

(2000, 48)

,TransactionID,TransactionAmt,ProductCD,card4,card6,isFraud,notebook_score
0,3485113,58.950,W,mastercard,credit,0,0.066462
1,3485114,39.000,W,visa,debit,0,0.099303
2,3485115,226.000,W,visa,debit,0,0.102744
3,3485116,5.841,C,mastercard,debit,0,0.010912
4,3485117,107.950,W,visa,debit,0,0.137042


In [3]:
def to_json(row):
    payment = row.drop(["isFraud", "notebook_score"]).to_dict()
    for key, value in payment.items():
        if pd.isna(value):
            payment[key] = None
        elif isinstance(value, np.generic):
            payment[key] = value.item()
    return payment

payments = [to_json(row) for _, row in sample.iterrows()]
payments[0]

{'TransactionID': 3485113,
 'TransactionDT': 13046456,
 'TransactionAmt': 58.95,
 'ProductCD': 'W',
 'card1': 4141,
 'card2': 404.0,
 'card3': 150.0,
 'card4': 'mastercard',
 'card5': 102.0,
 'card6': 'credit',
 'addr1': 441.0,
 'addr2': 87.0,
 'dist1': None,
 'P_emaildomain': 'hotmail.com',
 'R_emaildomain': None,
 'DeviceType': None,
 'DeviceInfo': None,
 'C1': 1.0,
 'C2': 1.0,
 'C3': 0.0,
 'C4': 0.0,
 'C5': 1.0,
 'C6': 1.0,
 'C7': 0.0,
 'C8': 0.0,
 'C9': 1.0,
 'C10': 0.0,
 'C11': 1.0,
 'C12': 0.0,
 'C13': 3.0,
 'C14': 1.0,
 'D1': 20.0,
 'D2': 20.0,
 'D3': 7.0,
 'D4': 20.0,
 'D10': 20.0,
 'D15': 20.0,
 'M1': None,
 'M2': None,
 'M3': None,
 'M4': None,
 'M5': None,
 'M6': 'F',
 'M7': None,
 'M8': None,
 'M9': None}

## 3. Start the API

This is the same command used to run it normally, from the project folder:

`uvicorn api.fraud_api:app --port 8030`

At startup it loads the model and replays the history up to day 150, so it knows each customer's past payments.

In [4]:
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "api.fraud_api:app", "--port", "8030", "--log-level", "warning"], cwd="..")

url = "http://127.0.0.1:8030"
for i in range(300):
    try:
        print(httpx.get(url + "/health").json())
        break
    except httpx.TransportError:
        time.sleep(1)

{'status': 'ok', 'model': 'xgboost', 'history_rows': 498113, 'warm_seconds': 22.7, 'customers': 190501}


## 4. Score one payment

In [5]:
httpx.post(url + "/score", json=payments[0]).json()

{'TransactionID': 3485113,
 'risk_score': 0.066462,
 'decision': 'APPROVE',
 'reasons': []}

Sending the same payment again should be rejected.

In [6]:
response = httpx.post(url + "/score", json=payments[0])
print(response.status_code, response.json())

409 {'detail': 'payment already scored'}


## 5. Score the rest and compare with the notebook

In [7]:
client = httpx.Client(timeout=30)
api_scores = [None]
decisions = []
reasons = []
times = []

for payment in payments[1:]:
    start = time.perf_counter()
    result = client.post(url + "/score", json=payment).json()
    times.append((time.perf_counter() - start) * 1000)
    api_scores.append(result["risk_score"])
    decisions.append(result["decision"])
    reasons.append(result["reasons"])

check = sample.iloc[1:].copy()
check["api_score"] = api_scores[1:]
check["decision"] = decisions
check["reasons"] = reasons

diff = (check["api_score"] - check["notebook_score"]).abs()
print("payments scored:", len(check))
print("biggest difference from notebook score:", diff.max())
print("same score (to 5 decimals):", round((diff < 0.00001).mean() * 100, 2), "%")

payments scored: 1999
biggest difference from notebook score: 4.998915195467135e-07
same score (to 5 decimals): 100.0 %


In [8]:
print("time per request (ms): average", round(np.mean(times), 1), "| p95", round(np.percentile(times, 95), 1))
check.groupby("decision")["isFraud"].agg(["count", "sum", "mean"])

time per request (ms): average 31.8 | p95 34.5


,count,sum,mean
decision,,,
APPROVE,1950,40,0.020513
BLOCK,9,9,1.000000
STEP_UP,40,24,0.600000


Some flagged payments and the reasons returned:

In [9]:
check[check["decision"] != "APPROVE"].sort_values("api_score", ascending=False)[
    ["TransactionID", "TransactionAmt", "api_score", "decision", "reasons", "isFraud"]].head(8)

,TransactionID,TransactionAmt,api_score,decision,reasons,isFraud
1773,3486886,31.724,0.998021,BLOCK,"[C1, C13, C8]",1
1763,3486876,31.724,0.997952,BLOCK,"[C1, C8, C13]",1
1750,3486863,63.447,0.996911,BLOCK,"[C1, C8, C13]",1
591,3485704,15.862,0.996782,BLOCK,"[C1, C8, C4]",1
594,3485707,15.862,0.996665,BLOCK,"[C1, C8, C4]",1
1770,3486883,31.724,0.996144,BLOCK,"[C1, C8, C13]",1
588,3485701,15.862,0.994970,BLOCK,"[C1, C8, C4]",1
808,3485921,6.517,0.994338,BLOCK,"[C1, C8, C4]",1


## 6. Stop the API

In [10]:
client.close()
server.terminate()
server.wait()
print("stopped")

stopped


## Notes
- While the API runs, `http://127.0.0.1:8030/docs` gives a page to try the endpoints.
- The history is kept in memory, so it is only for a demo. In production it would be in a store like Redis.